# Weight experiment — Qwen3-4B trên TPU v5e-8 (seed 123)

5 cấu hình: natural, balanced, no_desc, tiki_heavy, title_heavy.

- cùng 5M token, 625 optimizer step
- cùng eval set 1.050 dòng (1688 title/attr/desc + Tiki title/attr)
- chỉ khác phân bố token theo group


In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-U', 'transformers>=4.51', 'peft', 'kagglehub'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
               capture_output=True)
import torch, torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
print('torch', torch.__version__, '| xla', torch_xla.__version__)
print('world_size', xr.world_size())
print('device', xm.xla_device())


In [ ]:
from pathlib import Path
BUILDER = "#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\"\"\"Dựng eval set cố định và train view cho thực nghiệm weight.\n\nMục tiêu: mọi cấu hình weight và mọi seed nhìn đúng cùng một eval set,\ncùng id, cùng thứ tự; chỉ khác phân bố token lúc train.\n\nOutput:\n  data/cleaned/weight_eval/eval_1688.parquet\n  data/cleaned/weight_eval/eval_tiki.parquet\n  data/cleaned/weight_eval/train_view_4b.parquet\n  data/cleaned/weight_eval/train_view_1.7b.parquet\n  data/cleaned/weight_eval/meta.json\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\n\ndef target_tokens(row) -> float:\n    \"\"\"Ước lượng token phía target (ZH ~1.6 ký tự/token).\"\"\"\n    return float(row.get(\"tgt_chars\", 0) or 0) / 1.6\n\n\ndef make_eval_1688(df: pd.DataFrame, seed: int) -> pd.DataFrame:\n    test = df[df[\"split\"] == \"test\"].copy()\n    test[\"target_tokens\"] = test[\"target_text\"].map(\n        lambda x: len(str(x).replace(\" \", \"\")) / 1.6)\n    rows = []\n    quotas = {\"title\": 300, \"attribute\": 300, \"description\": 150}\n    for pair_type, quota in quotas.items():\n        part = test[test[\"pair_type\"] == pair_type]\n        part = part[(part[\"tokens_est\"] <= 1024) & (part[\"target_tokens\"] <= 700)]\n        n = min(quota, len(part))\n        rows.append(part.sample(n=n, random_state=seed))\n    out = pd.concat(rows, ignore_index=True)\n    out[\"dataset\"] = \"1688\"\n    out[\"eval_id\"] = [f\"1688v1:{i:05d}\" for i in range(len(out))]\n    out[\"max_new\"] = out[\"pair_type\"].map(\n        {\"title\": 64, \"attribute\": 256, \"description\": 768})\n    out[\"source\"] = out[\"source_text\"]\n    out[\"reference\"] = out[\"target_text\"]\n    cols = [\"eval_id\", \"dataset\", \"pair_type\", \"product_id\", \"source\",\n            \"reference\", \"max_new\", \"tokens_est\"]\n    return out[cols].reset_index(drop=True)\n\n\ndef make_eval_tiki(df: pd.DataFrame, seed: int) -> pd.DataFrame:\n    test = df[df[\"split\"] == \"test\"].copy()\n    test = test[test[\"qty_status\"].fillna(\"qty_ok\") == \"qty_ok\"]\n    rows = []\n    quotas = {\"title\": 300, \"attribute\": 300}\n    for pair_type, quota in quotas.items():\n        part = test[(test[\"pair_type\"] == pair_type) & (test[\"tokens_est\"] <= 1024)]\n        n = min(quota, len(part))\n        rows.append(part.sample(n=n, random_state=seed))\n    out = pd.concat(rows, ignore_index=True)\n    out[\"dataset\"] = \"tiki\"\n    out[\"eval_id\"] = [f\"tikiv1:{i:05d}\" for i in range(len(out))]\n    out[\"max_new\"] = out[\"pair_type\"].map({\"title\": 64, \"attribute\": 256})\n    out[\"source\"] = out[\"source_text\"]\n    # Tiki không có reference người: dùng COMET-QE, không dùng target_text.\n    out[\"reference\"] = \"\"\n    cols = [\"eval_id\", \"dataset\", \"pair_type\", \"product_id\", \"source\",\n            \"reference\", \"max_new\", \"tokens_est\"]\n    return out[cols].reset_index(drop=True)\n\n\ndef make_train_view(path: Path, max_seq: int) -> pd.DataFrame:\n    df = pd.read_parquet(path)\n    train = df[df[\"split\"] == \"train\"].copy()\n    train = train[(train[\"tokens_est\"] >= 2) & (train[\"tokens_est\"] <= max_seq)]\n    train = train[train[\"target_text\"].astype(str).str.strip().ne(\"\")]\n    if \"domain\" in train.columns:\n        train[\"group\"] = \"tiki\"\n        is_1688 = train[\"domain\"] == \"1688\"\n        train.loc[is_1688, \"group\"] = train.loc[is_1688, \"pair_type\"].map(\n            {\"title\": \"title_1688\", \"attribute\": \"attr_1688\",\n             \"description\": \"desc_1688\"})\n    else:\n        train[\"group\"] = train[\"pair_type\"].map(\n            {\"title\": \"title_1688\", \"attribute\": \"attr_1688\",\n             \"description\": \"desc_1688\"}).fillna(\"tiki\")\n    train[\"eff_tokens\"] = train[\"tokens_est\"].clip(lower=1).astype(float)\n    return train[[\"pair_id\", \"domain\", \"pair_type\", \"product_id\", \"group\",\n                  \"source_text\", \"target_text\", \"key_vi\", \"tokens_est\",\n                  \"eff_tokens\"]].reset_index(drop=True)\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--pairs\", default=\"data/cleaned/pairs_dedup\")\n    ap.add_argument(\"--out\", default=\"data/cleaned/weight_eval\")\n    ap.add_argument(\"--max-seq\", type=int, default=1024)\n    ap.add_argument(\"--seed\", type=int, default=20261104)\n    args = ap.parse_args()\n\n    pairs, out = Path(args.pairs), Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    d1688 = pd.read_parquet(pairs / \"1688.parquet\")\n    eval_1688 = make_eval_1688(d1688, args.seed)\n    eval_1688.to_parquet(out / \"eval_1688.parquet\", index=False)\n\n    views = {}\n    for model in (\"4b\", \"1.7b\"):\n        path = pairs / f\"tiki_qwen3-{model}.parquet\"\n        tiki = pd.read_parquet(path)\n        if model == \"4b\":\n            eval_tiki = make_eval_tiki(tiki, args.seed)\n            eval_tiki.to_parquet(out / \"eval_tiki.parquet\", index=False)\n        train = pd.concat([make_train_view(pairs / \"1688.parquet\", args.max_seq),\n                           make_train_view(path, args.max_seq)],\n                          ignore_index=True)\n        train.to_parquet(out / f\"train_view_{model}.parquet\", index=False)\n        mass = train.groupby(\"group\")[\"eff_tokens\"].sum()\n        views[model] = {\n            \"rows\": int(len(train)),\n            \"tokens\": round(float(mass.sum())),\n            \"group_tokens\": {k: round(float(v)) for k, v in mass.items()},\n            \"group_share\": {k: round(float(v / mass.sum()), 4) for k, v in mass.items()},\n            \"group_rows\": {k: int(v) for k, v in train.groupby(\"group\").size().items()},\n        }\n\n    meta = {\n        \"max_seq\": args.max_seq,\n        \"seed\": args.seed,\n        \"eval_1688\": {\n            \"rows\": int(len(eval_1688)),\n            \"by_type\": {k: int(v) for k, v in eval_1688.groupby(\"pair_type\").size().items()},\n        },\n        \"eval_tiki\": {\n            \"rows\": int(len(eval_tiki)),\n            \"by_type\": {k: int(v) for k, v in eval_tiki.groupby(\"pair_type\").size().items()},\n        },\n        \"views\": views,\n    }\n    (out / \"meta.json\").write_text(\n        json.dumps(meta, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    print(json.dumps(meta, ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
Path('build_weight_eval_sets.py').write_text(BUILDER, encoding='utf-8')
TRAINER = "#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\"\"\"Train Qwen3-4B LoRA trên TPU v5e-8 cho thực nghiệm weight.\n\nThiết kế công bằng:\n  - cùng eval set, cùng id, cùng prompt\n  - cùng token budget (5M) và cùng số optimizer step (625)\n  - mỗi step đúng ~8000 token, chia đều cho 8 chip\n  - chỉ khác phân bố token theo group (natural/balanced/...)\n\nChạy trong Kaggle TPU kernel:\n  python tpu_weight_train.py --seed 42 --out /kaggle/working/weight_runs\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nCONFIGS = {\n    \"natural\": None,\n    \"balanced\": {\"title_1688\": 0.15, \"attr_1688\": 0.35,\n                 \"desc_1688\": 0.25, \"tiki\": 0.25},\n    \"no_desc\": {\"title_1688\": 0.25, \"attr_1688\": 0.45,\n                \"desc_1688\": 0.00, \"tiki\": 0.30},\n    \"tiki_heavy\": {\"title_1688\": 0.10, \"attr_1688\": 0.25,\n                   \"desc_1688\": 0.20, \"tiki\": 0.45},\n    \"title_heavy\": {\"title_1688\": 0.30, \"attr_1688\": 0.30,\n                    \"desc_1688\": 0.15, \"tiki\": 0.25},\n}\nSYSTEM = (\"You are a professional Vietnamese-to-Simplified Chinese e-commerce translator. \"\n          \"Translate the user text into Simplified Chinese. Preserve numbers, units and brand names. \"\n          \"Return only the translation, no explanation.\")\nBUCKETS = (64, 128, 256, 512, 1024)\nMODEL_PATH = \"\"\nTRAIN = None\nEVAL_1688 = None\nEVAL_TIKI = None\nOUT_DIR = Path(\"/kaggle/working/weight_runs\")\nSEED = 42\nSTEPS = 625\nBUDGET = 5_000_000\nMAX_SEQ = 1024\nMICRO_BATCH = 2\nWORLD = 8\nSKIP_GENERATION = False\n\n\ndef log(message: str) -> None:\n    print(f\"[{time.strftime('%H:%M:%S')}] {message}\", flush=True)\n\n\ndef sample_indices(train: pd.DataFrame, config: dict | None, seed: int,\n                   budget: int) -> tuple[list[int], float]:\n    rng = np.random.default_rng(seed)\n    n = len(train)\n    eff = train[\"eff_tokens\"].to_numpy(dtype=float)\n    groups = train[\"group\"].to_numpy()\n    mass = pd.Series(eff).groupby(groups).sum()\n    if config is None:\n        weights = {g: 1.0 for g in mass.index}\n    else:\n        weights = {g: float(config.get(g, 0.0)) / max(1.0, float(mass[g]))\n                   for g in mass.index}\n    raw = np.array([weights.get(g, 0.0) for g in groups]) * eff\n    prob = raw / raw.sum()\n    chosen, total = [], 0.0\n    while total < budget:\n        draw = rng.choice(n, size=8192, p=prob)\n        stop = False\n        for idx in draw:\n            value = float(eff[idx])\n            if total + value > budget:\n                stop = True\n                break\n            chosen.append(int(idx))\n            total += value\n        if stop:\n            break\n    return chosen, total\n\n\ndef build_plan(train: pd.DataFrame, config: dict | None, seed: int,\n               budget: int, steps: int, world: int) -> tuple[list[list[int]], float]:\n    stream, total = sample_indices(train, config, seed, budget)\n    target = budget / steps\n    groups: list[list[int]] = []\n    current: list[int] = []\n    current_tokens = 0.0\n    cursor = 0\n    while len(groups) < steps:\n        if cursor >= len(stream):\n            cursor = 0\n        idx = stream[cursor]\n        cursor += 1\n        current.append(idx)\n        current_tokens += float(train[\"eff_tokens\"].iloc[idx])\n        if current_tokens >= target and len(current) >= world:\n            groups.append(current)\n            current, current_tokens = [], 0.0\n    if current and len(groups) == steps:\n        groups[-1].extend(current)\n    actual = sum(float(train[\"eff_tokens\"].iloc[idx]) for group in groups for idx in group)\n    return groups, actual\n\n\ndef assign_ranks(step: list[int], train: pd.DataFrame, world: int) -> list[list[int]]:\n    eff = train[\"eff_tokens\"].to_numpy(dtype=float)\n    bins: list[list[int]] = [[] for _ in range(world)]\n    loads = [0.0] * world\n    counts = [0] * world\n    for idx in sorted(step, key=lambda i: float(eff[i]), reverse=True):\n        rank = min(range(world), key=lambda r: (loads[r], counts[r], r))\n        bins[rank].append(int(idx))\n        loads[rank] += float(eff[idx])\n        counts[rank] += 1\n    return bins\n\n\ndef build_prompt(tokenizer, source: str) -> str:\n    messages = [{\"role\": \"system\", \"content\": SYSTEM},\n                {\"role\": \"user\", \"content\": str(source or \"\").strip()}]\n    try:\n        return tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True,\n            enable_thinking=False)\n    except TypeError:\n        return tokenizer.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True)\n\n\ndef encode_row(tokenizer, row, max_seq: int):\n    prompt = build_prompt(tokenizer, row[\"source_text\"])\n    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids\n    if len(prompt_ids) >= max_seq - 8:\n        return None\n    target = str(row[\"target_text\"] or \"\").strip()\n    if not target:\n        return None\n    full = tokenizer(prompt + target + tokenizer.eos_token,\n                     add_special_tokens=False).input_ids[:max_seq]\n    if len(full) <= len(prompt_ids) + 1:\n        return None\n    labels = [-100] * len(prompt_ids) + full[len(prompt_ids):]\n    return full, labels\n\n\ndef collate(items, pad_id: int):\n    length = max(len(x[0]) for x in items)\n    length = min([b for b in BUCKETS if b >= length] or [BUCKETS[-1]])\n    ids, labs, mask = [], [], []\n    for seq, lab in items:\n        pad = length - len(seq)\n        ids.append(seq + [pad_id] * pad)\n        labs.append(lab + [-100] * pad)\n        mask.append([1] * len(seq) + [0] * pad)\n    return {\"input_ids\": torch.tensor(ids, dtype=torch.long),\n            \"labels\": torch.tensor(labs, dtype=torch.long),\n            \"attention_mask\": torch.tensor(mask, dtype=torch.long)}\n\n\ndef generate_rows(model, tokenizer, frame: pd.DataFrame, rank: int, world: int,\n                  device, max_seq: int):\n    part = frame.iloc[rank::world].copy().reset_index(drop=True)\n    part[\"prompt\"] = [build_prompt(tokenizer, s) for s in part[\"source\"]]\n    part[\"prompt_tokens\"] = part[\"prompt\"].map(\n        lambda x: len(tokenizer(x, add_special_tokens=False).input_ids))\n    part = part[part[\"prompt_tokens\"] < max_seq - 8]\n    # gom theo bucket cố định để XLA chỉ compile vài shape, không compile mỗi batch\n    part[\"bucket\"] = part[\"prompt_tokens\"].map(\n        lambda n: min([b for b in BUCKETS if b >= max(16, int(n))] or [BUCKETS[-1]]))\n    out_map: dict[str, str] = {}\n    for (max_new, bucket), group in part.groupby([\"max_new\", \"bucket\"]):\n        records = group.sort_values(\"prompt_tokens\").to_dict(\"records\")\n        batch_size = 1 if int(max_new) > 300 else 4\n        for start in range(0, len(records), batch_size):\n            chunk = records[start:start + batch_size]\n            enc = tokenizer([r[\"prompt\"] for r in chunk], return_tensors=\"pt\",\n                            padding=\"max_length\", truncation=True,\n                            max_length=int(bucket)).to(device)\n            with torch.inference_mode():\n                gen = model.generate(\n                    **enc, max_new_tokens=int(max_new), do_sample=False,\n                    eos_token_id=tokenizer.eos_token_id,\n                    pad_token_id=tokenizer.pad_token_id)\n            texts = tokenizer.batch_decode(\n                gen[:, enc[\"input_ids\"].shape[1]:], skip_special_tokens=True)\n            for row, text in zip(chunk, texts):\n                out_map[row[\"eval_id\"]] = text.strip()\n    return [{\"eval_id\": eid, \"hypothesis\": out_map.get(eid, \"\")}\n            for eid in part[\"eval_id\"]]\n\n\ndef _mp_fn(index: int) -> None:\n    import torch_xla.core.xla_model as xm\n    import torch_xla.runtime as xr\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    rank, world = xr.global_ordinal(), xr.world_size()\n    device = xm.xla_device()\n    log(f\"rank {rank}/{world} device={device}\")\n\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"left\"\n\n    try:\n        base = AutoModelForCausalLM.from_pretrained(\n            MODEL_PATH, dtype=torch.bfloat16, trust_remote_code=True,\n            low_cpu_mem_usage=True)\n    except TypeError:\n        base = AutoModelForCausalLM.from_pretrained(\n            MODEL_PATH, torch_dtype=torch.bfloat16, trust_remote_code=True,\n            low_cpu_mem_usage=True)\n    base.gradient_checkpointing_enable()\n    base.enable_input_require_grads()\n    model = None\n\n    plan = json.loads((OUT_DIR / f\"plan_seed{SEED}.json\").read_text(encoding=\"utf-8\"))\n    runs = list(plan.keys())\n\n    for run_idx, run_name in enumerate(runs):\n        config_name, seed_text = run_name.split(\"::\")\n        seed = int(seed_text)\n        steps_data = plan[run_name]\n        # mọi rank phải khởi tạo LoRA giống nhau\n        torch.manual_seed(seed)\n        np.random.seed(seed)\n        random.seed(seed)\n\n        lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias=\"none\",\n                              task_type=\"CAUSAL_LM\", target_modules=\"all-linear\")\n        adapter_name = f\"a{run_idx}\"\n        if model is None:\n            model = get_peft_model(base, lora_cfg, adapter_name=adapter_name)\n        else:\n            model.add_adapter(adapter_name, lora_cfg)\n        model.set_adapter(adapter_name)\n        model.to(device)\n        model.train()\n        model.gradient_checkpointing_enable()\n        model.config.use_cache = False\n\n        params = [p for p in model.parameters() if p.requires_grad]\n        optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=0.0)\n        warmup = 20\n        scheduler = torch.optim.lr_scheduler.LambdaLR(\n            optimizer,\n            lambda s: min(1.0, (s + 1) / warmup) *\n            max(0.0, 0.5 * (1 + math.cos(math.pi * min(1.0, s / STEPS)))))\n\n        started = time.time()\n        total_tokens = 0.0\n        for step, step_indices in enumerate(steps_data):\n            optimizer.zero_grad(set_to_none=True)\n            local = assign_ranks(step_indices, TRAIN, world)[rank]\n            step_tokens = sum(float(TRAIN[\"eff_tokens\"].iloc[idx]) for idx in step_indices)\n            local_loss = 0.0\n            local_tokens = 0\n            for start in range(0, len(local), MICRO_BATCH):\n                micro = local[start:start + MICRO_BATCH]\n                items = []\n                for idx in micro:\n                    item = encode_row(tokenizer, TRAIN.iloc[idx], MAX_SEQ)\n                    if item is not None:\n                        items.append(item)\n                if not items:\n                    continue\n                batch = {k: v.to(device) for k, v in collate(items, tokenizer.pad_token_id).items()}\n                out = model(**batch)\n                n_tok = int((batch[\"labels\"] != -100).sum().item())\n                loss = out.loss * (n_tok / max(1.0, step_tokens))\n                loss.backward()\n                xm.mark_step()\n                local_loss += float(out.loss.item()) * n_tok\n                local_tokens += n_tok\n            xm.optimizer_step(optimizer, barrier=True)\n            scheduler.step()\n            xm.mark_step()\n            total_tokens += step_tokens\n            if step % 25 == 0 and rank == 0:\n                log(f\"{config_name} seed={seed} step {step}/{STEPS} \"\n                    f\"loss={local_loss / max(1, local_tokens):.4f} \"\n                    f\"tokens={total_tokens/1e6:.2f}M \"\n                    f\"min={(time.time()-started)/60:.1f}\")\n\n        if rank == 0:\n            adapter_dir = OUT_DIR / f\"adapter_{config_name}_seed{seed}\"\n            model.save_pretrained(adapter_dir)\n            log(f\"saved {adapter_dir}\")\n        xm.rendezvous(f\"save_{run_idx}\")\n\n        if SKIP_GENERATION:\n            if rank == 0:\n                (OUT_DIR / f\"run_{config_name}_seed{seed}.json\").write_text(\n                    json.dumps({\"config\": config_name, \"seed\": seed,\n                                \"steps\": len(steps_data), \"tokens\": round(total_tokens),\n                                \"minutes\": round((time.time()-started)/60, 1),\n                                \"generation\": \"skipped\"}, ensure_ascii=False, indent=2),\n                    encoding=\"utf-8\")\n            xm.rendezvous(f\"done_{run_idx}\")\n            model.delete_adapter(adapter_name)\n            gc.collect()\n            continue\n\n        # generation\n        model.eval()\n        model.gradient_checkpointing_disable()\n        model.config.use_cache = True\n        shard_dir = OUT_DIR / \"hyp_shards\"\n        shard_dir.mkdir(exist_ok=True)\n        for dataset_name, frame in ((\"1688\", EVAL_1688), (\"tiki\", EVAL_TIKI)):\n            records = generate_rows(model, tokenizer, frame, rank, world,\n                                    device, MAX_SEQ)\n            for record in records:\n                record[\"dataset\"] = dataset_name\n                record[\"config\"] = config_name\n                record[\"seed\"] = seed\n            pd.DataFrame(records).to_parquet(\n                shard_dir / f\"{config_name}_{seed}_{dataset}_r{rank}.parquet\",\n                index=False)\n        xm.rendezvous(f\"gen_{run_idx}\")\n\n        if rank == 0:\n            for dataset_name in (\"1688\", \"tiki\"):\n                parts = []\n                for r in range(world):\n                    path = shard_dir / f\"{config_name}_{seed}_{dataset_name}_r{r}.parquet\"\n                    if path.exists():\n                        parts.append(pd.read_parquet(path))\n                merged = pd.concat(parts, ignore_index=True).sort_values(\"eval_id\")\n                frame = (EVAL_1688 if dataset_name == \"1688\" else EVAL_TIKI)\n                merged = frame[[\"eval_id\", \"dataset\", \"pair_type\", \"product_id\", \"source\", \"reference\"]].merge(\n                    merged[[\"eval_id\", \"hypothesis\", \"config\", \"seed\"]], on=\"eval_id\", how=\"left\")\n                merged[\"hypothesis\"] = merged[\"hypothesis\"].fillna(\"\")\n                merged[\"dataset\"] = dataset_name\n                merged.to_parquet(\n                    OUT_DIR / f\"hyp_{config_name}_seed{seed}_{dataset_name}.parquet\",\n                    index=False)\n            log(f\"merged {config_name} seed={seed}\")\n\n        model.train()\n        model.config.use_cache = False\n        model.gradient_checkpointing_enable()\n        if rank == 0:\n            (OUT_DIR / f\"run_{config_name}_seed{seed}.json\").write_text(\n                json.dumps({\"config\": config_name, \"seed\": seed,\n                            \"steps\": len(steps_data), \"tokens\": round(total_tokens),\n                            \"minutes\": round((time.time()-started)/60, 1)},\n                           ensure_ascii=False, indent=2), encoding=\"utf-8\")\n        xm.rendezvous(f\"done_{run_idx}\")\n\n        # giải phóng adapter cũ, giữ base\n        model.delete_adapter(adapter_name)\n        gc.collect()\n\n    log(f\"rank {rank} finished\")\n\n\ndef find_model() -> str:\n    try:\n        import kagglehub\n        return kagglehub.model_download(\"qwen-lm/qwen-3/Transformers/4b/1\")\n    except Exception as error:\n        log(f\"kagglehub failed: {error}\")\n    from huggingface_hub import snapshot_download\n    return snapshot_download(\"Qwen/Qwen3-4B\")\n\n\ndef main() -> int:\n    global MODEL_PATH, TRAIN, EVAL_1688, EVAL_TIKI, OUT_DIR, SEED, STEPS, BUDGET, SKIP_GENERATION\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--eval-dir\", default=\"/kaggle/working/weight_eval\")\n    ap.add_argument(\"--out\", default=\"/kaggle/working/weight_runs\")\n    ap.add_argument(\"--seed\", type=int, default=42)\n    ap.add_argument(\"--budget\", type=int, default=5_000_000)\n    ap.add_argument(\"--steps\", type=int, default=625)\n    ap.add_argument(\"--configs\", nargs=\"+\", default=list(CONFIGS))\n    ap.add_argument(\"--skip-generation\", action=\"store_true\")\n    ap.add_argument(\"--dry-run\", action=\"store_true\")\n    ap.add_argument(\"--smoke\", action=\"store_true\")\n    args = ap.parse_args()\n\n    if args.smoke:\n        args.configs = [\"natural\"]\n        args.steps = 2\n        args.budget = 16_000\n        args.skip_generation = True\n        args.out = \"/kaggle/working/smoke_runs\"\n    SKIP_GENERATION = bool(args.skip_generation)\n    OUT_DIR = Path(args.out)\n    OUT_DIR.mkdir(parents=True, exist_ok=True)\n    SEED, STEPS, BUDGET = args.seed, args.steps, args.budget\n\n    TRAIN = pd.read_parquet(Path(args.eval_dir) / \"train_view_4b.parquet\")\n    TRAIN = TRAIN[TRAIN[\"eff_tokens\"] <= 960].reset_index(drop=True)\n    EVAL_1688 = pd.read_parquet(Path(args.eval_dir) / \"eval_1688.parquet\")\n    EVAL_TIKI = pd.read_parquet(Path(args.eval_dir) / \"eval_tiki.parquet\")\n\n    plan = {}\n    for config_name in args.configs:\n        groups, actual = build_plan(TRAIN, CONFIGS[config_name], SEED, BUDGET,\n                                    STEPS, WORLD)\n        plan[f\"{config_name}::{SEED}\"] = groups\n        log(f\"plan {config_name}: {len(groups)} steps, {actual/1e6:.2f}M tokens\")\n    (OUT_DIR / f\"plan_seed{SEED}.json\").write_text(\n        json.dumps(plan, ensure_ascii=False), encoding=\"utf-8\")\n    if args.dry_run:\n        return 0\n\n    MODEL_PATH = find_model()\n    import torch_xla.distributed.xla_multiprocessing as xmp\n    xmp.spawn(_mp_fn, args=(), nprocs=WORLD, start_method=\"fork\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
Path('tpu_weight_train.py').write_text(TRAINER, encoding='utf-8')
print('scripts written')


In [ ]:
import glob
from pathlib import Path
hits = glob.glob('/kaggle/input/**/1688.parquet', recursive=True)
assert hits, 'không thấy 1688.parquet'
pairs = Path(hits[0]).parent
print('pairs dir', pairs)
import subprocess, sys
subprocess.run([sys.executable, 'build_weight_eval_sets.py',
                '--pairs', str(pairs),
                '--out', '/kaggle/working/weight_eval',
                '--max-seq', '1024'], check=True)


## Smoke 8 chip: 2 step, natural, không generate


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'tpu_weight_train.py',
                '--eval-dir', '/kaggle/working/weight_eval',
                '--out', '/kaggle/working/smoke_runs',
                '--smoke'], check=True)
print('SMOKE OK')


## Full run seed 123: 5 cấu hình × 5M token


In [ ]:
import subprocess, sys, time
started = time.time()
subprocess.run([sys.executable, 'tpu_weight_train.py',
                '--eval-dir', '/kaggle/working/weight_eval',
                '--out', '/kaggle/working/weight_runs',
                '--seed', '123'], check=True)
print('DONE in', round((time.time()-started)/60, 1), 'min')


In [ ]:
from pathlib import Path
for p in sorted(Path('/kaggle/working/weight_runs').glob('hyp_*.parquet')):
    print(p.name, p.stat().st_size)
print('--- adapters ---')
for p in sorted(Path('/kaggle/working/weight_runs').glob('adapter_*')):
    print(p.name)
